# 04 - Feature Engineering

## Objective

This notebook creates simple model features while keeping `quantity_sold` as the target. Historical demand features use only values before each prediction date to avoid data leakage.

**Input:** `data/processed/daily_product_sales.csv`  
**Output:** `data/processed/modeling_dataset.csv`  
**Next notebook:** `05_model_training.ipynb`

## Imports and Load Dataset

In [1]:
import os
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_PATH = ROOT / "data" / "processed" / "daily_product_sales.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["sale_date"])
df = df.sort_values(["product_name", "sale_date"]).reset_index(drop=True)
print("Shape:", df.shape)

Shape: (6168, 11)


## Create Date Features

In [2]:
df_features = df.copy()
df_features["day_of_week"] = df_features["sale_date"].dt.dayofweek
df_features["day_of_month"] = df_features["sale_date"].dt.day
df_features["month"] = df_features["sale_date"].dt.month
df_features["week_of_year"] = df_features["sale_date"].dt.isocalendar().week.astype(int)
df_features["is_weekend"] = df_features["day_of_week"].isin([5, 6]).astype(int)
df_features.head()

,product_name,sale_date,quantity_sold,total_revenue_brl,estimated_profit_brl,discount_pct,unit_price_brl,current_stock_snapshot,supplier_lead_time_days,product_category,weather_condition,day_of_week,day_of_month,month,week_of_year,is_weekend
0,Bag Delivery 45L,2025-01-01,5,889.97,414.97,7.5,178.1725,101.0,3.0,Bag Entrega,Ensolarado,2,1,1,1,0
1,Bag Delivery 45L,2025-01-02,2,364.80,174.80,0.0,182.4000,53.0,7.0,Bag Entrega,Ensolarado,3,2,1,1,0
2,Bag Delivery 45L,2025-01-03,1,183.92,88.92,5.0,183.9200,24.0,7.0,Bag Entrega,Ensolarado,4,3,1,1,0
3,Bag Delivery 45L,2025-01-04,3,536.25,251.25,5.0,178.7500,52.0,7.0,Bag Entrega,Ensolarado,5,4,1,1,1
4,Bag Delivery 45L,2025-01-05,8,1442.82,682.82,4.0,179.0000,109.0,8.0,Bag Entrega,Ensolarado,6,5,1,1,1


## Create Lag and Rolling Features

Lag features are created separately for each product. Rolling means use `shift(1)` first, so the current target is never included in its own features.

In [3]:
grouped_demand = df_features.groupby("product_name")["quantity_sold"]
grouped_price = df_features.groupby("product_name")["unit_price_brl"]

df_features["unit_price_1d"] = grouped_price.shift(1)
df_features["quantity_1d"] = grouped_demand.shift(1)
df_features["quantity_7d"] = grouped_demand.shift(7)
df_features["quantity_14d"] = grouped_demand.shift(14)

for window in [1, 7, 14]:
    df_features[f"rolling_{window}d"] = grouped_demand.transform(
        lambda values, window=window: values.shift(1).rolling(window).mean()
    )

df_features[["product_name", "sale_date", "quantity_sold", "quantity_1d", "quantity_7d", "quantity_14d", "rolling_7d", "rolling_14d"]].head(16)

,product_name,sale_date,quantity_sold,quantity_1d,quantity_7d,quantity_14d,rolling_7d,rolling_14d
0,Bag Delivery 45L,2025-01-01,5,NaN,NaN,NaN,NaN,NaN
1,Bag Delivery 45L,2025-01-02,2,5.0,NaN,NaN,NaN,NaN
2,Bag Delivery 45L,2025-01-03,1,2.0,NaN,NaN,NaN,NaN
3,Bag Delivery 45L,2025-01-04,3,1.0,NaN,NaN,NaN,NaN
4,Bag Delivery 45L,2025-01-05,8,3.0,NaN,NaN,NaN,NaN
5,Bag Delivery 45L,2025-01-06,2,8.0,NaN,NaN,NaN,NaN
6,Bag Delivery 45L,2025-01-07,5,2.0,NaN,NaN,NaN,NaN
7,Bag Delivery 45L,2025-01-08,6,5.0,5.0,NaN,3.714286,NaN
8,Bag Delivery 45L,2025-01-09,3,6.0,2.0,NaN,3.857143,NaN
9,Bag Delivery 45L,2025-01-10,2,3.0,1.0,NaN,4.000000,NaN


## Handle Missing History

The first 14 rows of each product do not have enough past data. They are removed instead of filling unknown lags with zero. The previous-day price can also be missing near the start of a product history, so those rows are removed as well.

In [4]:
history_features = [
    "quantity_1d", "quantity_7d", "quantity_14d",
    "rolling_1d", "rolling_7d", "rolling_14d",
]

rows_before = len(df_features)
df_features = df_features.dropna(subset=history_features + ["unit_price_1d"]).copy()
print("Rows removed because history or known business values were unavailable:", rows_before - len(df_features))

Rows removed because history or known business values were unavailable: 168


## Select Modeling Columns

The same-day average transaction price is not available before the day is complete. The model therefore uses `unit_price_1d`, which is the previous known daily price for each product.

Weather remains in the prepared dataset for analysis, but it is not used as a model feature because this project does not have future weather forecasts. Revenue, profit, discount, stock and supplier lead time are also excluded. Revenue and profit are same-day outcomes, while stock and lead time belong to the later recommendation rules.

In [5]:
target = "quantity_sold"
numeric_features = [
    "unit_price_1d", "day_of_week", "day_of_month", "month",
    "week_of_year", "is_weekend", "quantity_1d", "quantity_7d",
    "quantity_14d", "rolling_1d", "rolling_7d", "rolling_14d",
]
categorical_features = ["product_name"]

modeling_columns = ["sale_date", target] + numeric_features + categorical_features
df_features = df_features[modeling_columns].sort_values(["sale_date", "product_name"]).reset_index(drop=True)

## Validate and Save Dataset

In [6]:
assert not df_features.duplicated(["product_name", "sale_date"]).any()
assert df_features[modeling_columns].notna().all().all()
assert (df_features[target] >= 0).all()
assert target not in numeric_features + categorical_features

OUTPUT_PATH = ROOT / "data" / "processed" / "modeling_dataset.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_features.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Shape:", df_features.shape)
print("Date range:", df_features["sale_date"].min(), "to", df_features["sale_date"].max())
print("Saved:", OUTPUT_PATH.resolve())
df_features.head()

Shape: (6000, 15)
Date range: 2025-01-15 00:00:00 to 2026-05-29 00:00:00
Saved: C:\DEV\motostock-ai\data\processed\modeling_dataset.csv


,sale_date,quantity_sold,unit_price_1d,day_of_week,day_of_month,month,week_of_year,is_weekend,quantity_1d,quantity_7d,quantity_14d,rolling_1d,rolling_7d,rolling_14d,product_name
0,2025-01-15,5,178.557500,2,15,1,3,0,7.0,6.0,5.0,7.0,4.428571,4.071429,Bag Delivery 45L
1,2025-01-15,0,268.105000,2,15,1,3,0,2.0,1.0,3.0,2.0,1.714286,2.428571,Bag Delivery 80L
2,2025-01-15,1,35.605000,2,15,1,3,0,3.0,0.0,0.0,3.0,1.285714,0.928571,Balaclava
3,2025-01-15,1,261.635000,2,15,1,3,0,2.0,1.0,0.0,2.0,2.000000,1.428571,Baú Moto 30L
4,2025-01-15,3,99.583333,2,15,1,3,0,6.0,5.0,0.0,6.0,5.571429,3.285714,Capa de Chuva


## Summary

The modeling dataset contains calendar features, the previous known product price and product-level demand history for 1, 7 and 14 days. Rolling features use only past values. Same-day price and weather are not used, and missing history was removed rather than replaced with invented demand.

The next notebook uses the same feature list and a full-date temporal split to train Random Forest and XGBoost pipelines.